# RENT PREDICTION APPLICATION

In [1]:
import os
from os.path import split

import pandas as pd
import numpy as np

In [2]:
def drop_columns(df , clmns):
    """
    This is a function to drop columns from a dataframe.
    """
    for col in clmns:
        try:
            df.drop([col], axis=1, inplace=True)
            print(f"Column {col} dropped successfully.")
        except Exception as e :
            print(f"An exception occurred while dropping {col}: {e}")


In [3]:
files = os.listdir('hepsiemlak')
df = pd.concat((pd.read_csv(f'hepsiemlak/{file}') for file in files), ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1609 entries, 0 to 1608
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   list-view-price     1609 non-null   object
 1   celly               1609 non-null   str   
 2   celly 2             1609 non-null   str   
 3   celly 3             1609 non-null   str   
 4   celly 4             1576 non-null   str   
 5   list-view-location  1609 non-null   str   
dtypes: object(1), str(5)
memory usage: 75.6+ KB


In [4]:
df['city'] = df['list-view-location'].apply(lambda x: x.split('/')[0])
df['district'] = df['list-view-location'].apply(lambda x: x.split('/')[1])
df['neighborhood'] = df['list-view-location'].apply(lambda x: x.split('/')[2])

In [5]:
df.rename(columns={'celly':'rooms',
                   'celly 2':'area_mSqr',
                   'celly 3':'building_age',
                   'celly 4':'floor'}, inplace=True)

In [6]:
df['rooms'] = df['rooms'].apply(lambda x: x.replace('Stüdyo','1+0'))
df['rooms'] = df['rooms'].str.replace("\n", "", regex=False)
df['room'] = df['rooms'].str.split('+', expand=True)[0].astype(int)
df['living_room'] = df['rooms'].str.split('+', expand=True)[1].astype(int)
cols = ['rooms','list-view-location']
drop_columns(df, cols)

Column rooms dropped successfully.
Column list-view-location dropped successfully.


In [7]:
df['building_age'] = df['building_age'].str.replace("\n", "", regex=False)
df['building_age'] = df['building_age'].str.replace("Yaşında", "", regex=False)
df['building_age'] = df['building_age'].str.replace("at Age", "", regex=False)
df['building_age'] = df['building_age'].str.replace("Sıfır Bina", "0", regex=False)
df['building_age'] = df['building_age'].str.replace("New Property", "0", regex=False)



In [8]:
df['area_mSqr'] = df['area_mSqr'].str.replace("m²", "", regex=False)

In [29]:
df['floor'] = df['floor'].str.replace("Kat", "Floor", regex=False)
df['floor'] = df['floor'].str.replace("Ara Floor", "Middle  Floor", regex=False)
df['floor'] = df['floor'].str.replace('Middle  Floor', 'Middle Floor', regex=False)
df['floor'] = df['floor'].str.replace("Floorı", "Floor", regex=False)
df['floor'] = df['floor'].str.replace("Giriş", "0.", regex=False)
df['floor'] = df['floor'].str.replace("Zemin", "0. Floor")
df['floor'] = df['floor'].str.replace("Ground Floor", "0. Floor")
df['floor'] = df['floor'].str.replace("Undergroung 1", "-1. Floor")
df['floor'] = df['floor'].str.replace("Underground 2", "-2. Floor")
df['floor'] = df['floor'].str.replace("Bahçe", "Garden")
df['floor'] = df['floor'].str.replace("Raised 0. Floor", "Partially Basement")
df['floor'] = df['floor'].str.replace("Yüksek", "")
df['floor'] = df['floor'].str.replace("Basement", "Basement")
df['floor'] = df['floor'].str.replace("Kot 1", "1. Floor")
df['floor'] = df['floor'].str.replace("Underground", "-")
df['floor'] = df['floor'].str.replace("Çatı", "Top")
df['floor'] = df['floor'].str.replace("En Üst", "Top")
df['floor'] = df['floor'].str.replace("ve üzeri", "+")

df['floor'] = df['floor'].str.replace('.','')
df['floor'] = df['floor'].str.replace('Floor','')
df['floor'] = df['floor'].str.replace('Kot 2',"-2")
df['floor'] = df['floor'].str.replace('Groud','0')
df['floor'] = df['floor'].str.replace('Garden','0')
df['floor'] = df['floor'].str.replace('Basement','0')
df['floor'] = df['floor'].str.replace('Top','5')
df['floor'] = df['floor'].str.replace('Partially Basement','1')
df['floor'] = df['floor'].str.replace('Middle','3')
df['floor'] = df['floor'].str.replace('Partially 0','1')
df['floor'] = df['floor'].str.replace('Ground','0')
df['floor'] = df['floor'].str.replace('Underground','-1')
df['floor'] = df['floor'].str.replace('Villa','3')
df['floor'] = df['floor'].str.replace('Tripleks','3')
df['floor'] = df['floor'].str.replace(' ','')
df['floor'] = df['floor'].str.replace('+','')


In [51]:
df['list-view-price']=df['list-view-price'].astype(str)
df['list-view-price']=df['list-view-price'].apply(lambda x: x.replace(".", ""))
df['list-view-price']=df['list-view-price'].astype(int)
df.loc[df['list-view-price'] < 1000, 'list-view-price'] = (
    df.loc[df['list-view-price'] < 1000, 'list-view-price'] * 100
)
df['room'] = df['room'].astype(int)
df['list-view-price']=df['list-view-price'].astype(int)
df['living_room'] = df['living_room'].astype(int)
df['area_mSqr'] = df['area_mSqr'].astype(float)
df['building_age'] = df['building_age'].astype(int)
df['floor'] = df['floor'].astype(float)


In [45]:
try:
    df["rooms"] = df["rooms"].str.replace(" ", "", regex=False)
except Exception as e:
    print(f"An exception occurred:{e}")
    df = df[
        [
            'city',
            'district',
            'neighborhood',
            'room',
            'living_room',
            'area_mSqr',
            'building_age',
            'floor',
            'list-view-price'
        ]
    ]

df.to_csv("processed_data.csv", index=False)

An exception occurred:'rooms'
